Here, we preprocess the [immune dictionary](https://doi.org/10.1038/s41586-023-06816-9) dataset as described in scPerturb in anticipation that datasets from scPerturb will also be used. rds files are downloaded from the Immune Dictionary [download](https://www.immune-dictionary.org/app/home) page.

In [12]:
import os

import scanpy as sc
import anndata as ad

import pandas as pd
import numpy as np
from scipy.stats import pearsonr
from sklearn.metrics import normalized_mutual_info_score as nmi
import omnipath as op

import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings(action='ignore', module='pandas')
warnings.filterwarnings(action='ignore', category=FutureWarning, module='ipykernel')
warnings.filterwarnings(action='ignore', category=FutureWarning, module='scanpy')
warnings.filterwarnings(action='ignore', category=UserWarning, module='scanpy')

import sys
sclembas_path = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas_path))
from scLEMBAS.preprocess import get_tf_activity, embed_tf_activity
from scLEMBAS import io

In [3]:
n_cores = 12
os.environ["OMP_NUM_THREADS"] = str(n_cores)
os.environ["MKL_NUM_THREADS"] = str(n_cores)
os.environ["OPENBLAS_NUM_THREADS"] = str(n_cores)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(n_cores)
os.environ["NUMEXPR_NUM_THREADS"] = str(n_cores)

seed = 888
data_path = '/nobackup/users/hmbaghda/scLEMBAS/analysis'

In [4]:
quick_run = True # whether to subsample data and use quicker/less mem intensive (less accurate) versions of parameters

quick_dict = {'quick': {'perm': int(10), 'n_samples': int(1e3), 'batch_size': int(1e3)},
             'full': {'perm': int(1e3), 'n_samples': None, 'batch_size': int(1e4)}}
if quick_run: 
    run_key = 'quick'
else:
    run_key = 'full'

In [5]:
# grid search params
use_raw = False
# impute = True

In [6]:
directory_names = [
    #data_path,
    #os.path.join(data_path, 'raw'), os.path.join(data_path, 'raw', 'immune_dictionary'),
    os.path.join(data_path, 'interim'), os.path.join(data_path, 'interim', 'immune_dictionary_h5ad'),
    os.path.join(data_path, 'processed'), os.path.join(data_path, 'figures')
]

for directory_name in directory_names:
    if not os.path.exists(directory_name):
        os.makedirs(directory_name)


# Load Files

# Get TF activity estimates:

In [8]:
adata = sc.read_h5ad(os.path.join(data_path, 'processed', 'id_umap_all.h5ad'))

In [9]:
if quick_run:
    adata = sc.pp.subsample(adata, n_obs = quick_dict[run_key]['n_samples'], copy = True, random_state = seed) 

In [10]:
kwargs = {'args' : {'wsum' : {'times': quick_dict[run_key]['perm'], 'batch_size': quick_dict[run_key]['batch_size']},
                       'ulm' : {'batch_size': quick_dict[run_key]['batch_size']}, 
                        'mlm': {'batch_size': quick_dict[run_key]['batch_size']}
                       }}

In [14]:
adata = get_tf_activity(adata = adata, organism = 'mouse', grn = 'collectri', 
                                   verbose = True, min_n = 5,
                                   use_raw = False, # often set to False in the decoupler tutorials, despite default being True
                                   filter_pvals = False, pval_thresh = 1, hvg = False, static = True, 
                                  consensus = True, **kwargs)

Running scores.
15935 features of mat are empty, they will be removed.
Running mlm on mat with 1000 samples and 15118 targets for 602 sources.


100%|███████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.98s/it]


15935 features of mat are empty, they will be removed.
Running ulm on mat with 1000 samples and 15118 targets for 602 sources.


100%|███████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.75it/s]


15935 features of mat are empty, they will be removed.
Running wsum on mat with 1000 samples and 15118 targets for 602 sources.


100%|███████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.18s/it]


In [16]:
adata.obsm['consensus_estimate']

,Abl1,Aebp1,Ahr,Aire,Apex1,Ar,Arid1a,Arid3a,Arid3b,Arid4a,...,Zic1,Zkscan7,Zmynd8,Znf143,Znf148,Znf263,Znf354c,Znf382,Znf76,Zxdc
TCAGGTATCACGAACT-17,0.303695,-0.501479,-0.345414,-0.782284,0.515798,2.718550,-0.696311,-0.560636,-0.716401,-0.683835,...,1.567855,0.449749,1.037637,0.094317,-0.247426,-0.787314,-0.784729,-1.063950,1.395687,-0.790423
AACCTGAAGCACCAGA-21,0.587969,0.092327,0.412490,1.351518,0.555150,1.651813,-0.974656,-0.944613,-0.065148,0.117915,...,1.189800,0.155696,-0.127090,-0.515320,-1.413842,-0.874260,-0.311231,-0.932276,1.388268,-1.315244
TGTTCCGCAGACCTAT-32,NaN,-0.806521,NaN,NaN,NaN,NaN,-0.224127,-0.605030,-0.683384,-0.838494,...,NaN,-0.754792,NaN,-1.129944,-0.307753,-0.467932,NaN,-4.353477,-0.142986,NaN
CAGCAGCAGATCGGTG-06,-0.485380,0.003783,0.798099,1.108946,0.905811,1.726510,-1.287188,-0.819158,0.252358,0.621234,...,1.127686,-1.102080,-0.218124,0.095548,-1.516143,-0.162781,-0.498737,-0.088681,1.568918,-0.727687
ATCGTCCAGACGGATC-33,-0.235221,-0.514956,NaN,-0.844653,NaN,NaN,-0.438544,-0.759054,-0.823975,NaN,...,-0.453325,-0.583715,NaN,-0.574414,-1.492216,-0.607189,-1.008528,-1.662347,NaN,-1.738723
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
AGGAGGTAGGCCTTGC-38,0.320946,-0.603466,-1.091610,0.033380,1.162816,0.170637,-0.879044,-0.623879,-0.874640,-0.767197,...,1.284778,-0.679114,0.038305,-0.907857,0.243234,-0.022712,-1.349620,0.155607,-0.011914,-0.598371
AAAGTCCCACCTTCCA-26,NaN,-0.391968,NaN,NaN,NaN,NaN,-0.953159,-1.001703,-0.788334,-1.338839,...,NaN,-0.958403,NaN,NaN,-0.840885,NaN,NaN,NaN,0.010330,-1.125243
CTCAGGGCAAAGCACG-25,2.138365,-0.502067,0.263298,0.052768,0.254085,0.745530,-1.444483,-0.895553,0.936875,1.164220,...,1.778023,-0.453202,-0.356309,-0.956591,-0.991273,-0.990272,0.571034,-2.922416,-0.137249,-0.590255
CAACAGTAGATTAGTG-26,0.093774,-0.637578,0.259998,-0.519237,1.327173,1.394819,-1.294290,-0.130834,0.268253,-0.755107,...,0.237380,-0.073144,-0.586507,0.064840,-1.163934,0.078054,-0.018108,-1.259334,0.570574,-0.094838
